# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hashim123132/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

**Lane 1: Ranking Signal Analysis** — built on the warehouse via DuckDB.

Two signals checked first, then ONE rule encoded, ranked, and reviewed.

## Setup — connect to the warehouse

All queries run against the **`fact_content_query_90d`** table on Hugging Face. This table has pre-computed90-day aggregations per query, which we aggregate to page level.

In [1]:
%pip -q install duckdb huggingface_hub pandas numpy

import os, getpass, duckdb, numpy as np, pandas as pd

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
QUERY_90D = f"read_parquet('{REL}/fact_content_query_90d.parquet')"
print('Connected to warehouse.')

Note: you may need to restart the kernel to use updated packages.
Connected to warehouse.


## Load and aggregate to page level

The90d table has one row per query. We aggregate to one row per page by summing impressions/clicks and computing weighted-average position.

In [2]:
df = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(impressions_90d)                        AS impressions_90d,
        SUM(clicks_90d)                             AS clicks_90d,
        SUM(avg_position_90d * impressions_90d)
            / NULLIF(SUM(impressions_90d), 0)       AS avg_position,
        MAX(content_visible_query_count)             AS visible_queries,
        SUM(impressions_last30)                     AS impressions_last30,
        SUM(impressions_prev30)                     AS impressions_prev30
    FROM {QUERY_90D}
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(impressions_90d) >= 100
""").df()

df['ctr'] = (df.clicks_90d / df.impressions_90d * 100).round(2)
df['down'] = (
    (df.impressions_last30 < df.impressions_prev30 * 0.8)
    & (df.impressions_prev30 > 0)
).astype(int)

def position_tier(pos):
    if pd.isna(pos) or pos <= 0: return 'no_data'
    if pos <= 3: return 'top_3'
    if pos <= 10: return 'page_1'
    if pos <= 20: return 'striking'
    if pos <= 50: return 'page_3_5'
    return 'deep'

df['position_tier'] = df.avg_position.apply(position_tier)

print(f'rows: {len(df):,} | clients: {df.client_hash_id.nunique()}')
print(f'base rate (down): {df.down.mean():.3f}')

rows: 81,167 | clients: 50
base rate (down): 0.541


## 1. My rule and its reason codes

Two signals are checked first at least one must be behind a real FlyRank flag.

### Signal A — Volume behind quick-win flags

FlyRank's quick-win flags target low-volume pages where small improvements can move the needle. I test whether low-impression pages have higher decline rates.

In [5]:
# Signal A: volume vs decline rate (bucket table with n)
vA = df[df.impressions_90d > 0].copy()
vA['imp_tier'] = pd.cut(
    vA.impressions_90d,
    bins=[0, 500, 3000, 30000, 1e9],
    labels=['low', 'moderate', 'good', 'excellent']
)
tA = vA.groupby('imp_tier', observed=True).agg(
    n=('down', 'size'),
    down_rate=('down', 'mean')
).round(3)
print('Signal A: down rate by impression tier')
print(tA)

Signal A: down rate by impression tier
               n  down_rate
imp_tier                   
low        40660      0.523
moderate   28030      0.556
good       11408      0.568
excellent   1069      0.512


**Verdict: MIXED.** If low-impression pages show a higher decline rate, that supports the quick-win logic (volume as a signal). If the relationship is flat or inverted, the quick-win flag's volume assumption does not hold in this slice.

### Signal B — CTR relative to position

FlyRank's CTR-fix logic flags pages whose CTR is low for their ranking position. I test whether CTR varies systematically with `position_tier`.

In [6]:
# Signal B: median CTR by position tier (bucket table with n)
vB = df[(df.impressions_90d >= 500) & (df.avg_position > 0)].copy()
tB = vB.groupby('position_tier').agg(
    n=('ctr', 'size'),
    median_ctr=('ctr', 'median')
).round(3)
print('Signal B: median CTR by position_tier (impressions >= 500, avg_position > 0)')
print(tB)

Signal B: median CTR by position_tier (impressions >= 500, avg_position > 0)
                   n  median_ctr
position_tier                   
deep            2705        0.00
page_1         19424        0.17
page_3_5        7800        0.00
striking        9092        0.11
top_3           1516        0.17


**Verdict: CONFIRMED.** CTR is higher for pages in stronger ranking tiers. This supports looking for pages whose CTR is below the typical CTR for their ranking tier.

### Rule in plain words

Prioritize pages that have enough search visibility (>= 500 impressions), rank in a visible position (avg_position <= 20), and have a CTR below the median CTR for their position tier.

**Reason code:** `decline_risk_visible_page`

**Action label:** `review`

## 2. Build the ranked queue (writes the CSV)

**Score** = `(avg_position <= 20)` x `(tier median CTR - page CTR)` clipped at 0 x `log1p(impressions_90d)`.

Row gates: `impressions_90d >= 500` and `avg_position > 0`.
Inputs are all observed signals, knowable before any decision: impressions, position, CTR. No label, no future window.

In [7]:
v = df[(df.impressions_90d >= 500) & (df.avg_position > 0)].copy()
tier_ctr = v.groupby('position_tier')['ctr'].transform('median')
v['gap'] = (tier_ctr - v['ctr']).clip(lower=0)
v['score'] = (v.avg_position <= 20).astype(int) * v.gap * np.log1p(v.impressions_90d)

v['reason_code'] = 'decline_risk_visible_page'
v['action'] = 'review'

queue = v.sort_values('score', ascending=False)[[
    'content_hash_id', 'client_hash_id', 'position_tier', 'avg_position', 'ctr',
    'impressions_90d', 'clicks_90d', 'gap', 'score', 'reason_code', 'action'
]].copy()

os.makedirs('work/outputs', exist_ok=True)
out_path = 'work/outputs/baseline_action_score.csv'
queue.to_csv(out_path, index=False)
print('wrote', out_path, '| rows:', len(queue), '| flagged (score>0):', int((queue.score > 0).sum()))

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

labels = v.loc[queue.index, 'down'].to_numpy()
print('base rate (random pick):', round(labels.mean(), 3))
print('precision@10:', round(precision_at_k(queue.score, labels, 10), 3))
print('precision@50:', round(precision_at_k(queue.score, labels, 50), 3))

wrote work/outputs/baseline_action_score.csv | rows: 40537 | flagged (score>0): 14655
base rate (random pick): 0.558
precision@10: 0.5
precision@50: 0.56


## 3. Top-10 review

For each of the top 10: the action, why it is there, and what would make it wrong.

In [8]:
top10 = queue.head(10)
for i, (_, r) in enumerate(top10.iterrows(), 1):
    print(f"{i:2d}. {r['content_hash_id'][:24]}  tier={r['position_tier']}  "
          f"pos={r['avg_position']:.1f}  ctr={r['ctr']:.2f}  "
          f"imp={r['impressions_90d']:,}  score={r['score']:.2f}")

 1. content_39e19a3ec2d95f9d  tier=page_1  pos=9.8  ctr=0.00  imp=379,229.0  score=2.18
 2. content_11bf4c33adea7bdc  tier=page_1  pos=8.4  ctr=0.00  imp=332,312.0  score=2.16
 3. content_012de75c008aa653  tier=page_1  pos=8.2  ctr=0.00  imp=253,583.0  score=2.12
 4. content_65c75874a23fca87  tier=page_1  pos=8.1  ctr=0.00  imp=219,689.0  score=2.09
 5. content_23a42776a7009b65  tier=page_1  pos=9.7  ctr=0.00  imp=203,573.0  score=2.08
 6. content_8e1334d6356668e3  tier=page_1  pos=3.2  ctr=0.00  imp=158,016.0  score=2.03
 7. content_425715547c6a3ea8  tier=page_1  pos=8.6  ctr=0.00  imp=143,398.0  score=2.02
 8. content_d0acf7062bc6b257  tier=top_3  pos=2.1  ctr=0.01  imp=297,755.0  score=2.02
 9. content_9610ee5585b7a7fc  tier=page_1  pos=9.7  ctr=0.00  imp=138,391.0  score=2.01
10. content_33d31496fca9665e  tier=page_1  pos=6.2  ctr=0.01  imp=273,776.0  score=2.00


| # | content | tier | pos | CTR | impressions | why it is there | what would make it wrong |
|---|---|---|---|---|---|---|---|
| 1 | content_39e19a3ec2d95f9d | page_1 | 9.8 | 0.00 | 379,229 | CTR far below tier median at visible position | Intent mismatch, not decay; SERP change |
| 2 | content_11bf4c33adea7bdc | page_1 | 8.4 | 0.00 | 332,312 | CTR far below tier median at visible position | Intent mismatch, not decay; SERP change |
| 3 | content_012de75c008aa653 | page_1 | 8.2 | 0.00 | 253,583 | CTR far below tier median at visible position | Intent mismatch, not decay; SERP change |
| 4 | content_65c75874a23fca87 | page_1 | 8.1 | 0.00 | 219,689 | CTR far below tier median at visible position | Intent mismatch, not decay; SERP change |
| 5 | content_23a42776a7009b65 | page_1 | 9.7 | 0.00 | 203,573 | CTR far below tier median at visible position | Intent mismatch, not decay; SERP change |
| 6 | content_8e1334d6356668e3 | page_1 | 3.2 | 0.00 | 158,016 | CTR far below tier median at visible position | Intent mismatch, not decay; SERP change |
| 7 | content_425715547c6a3ea8 | page_1 | 8.6 | 0.00 | 143,398 | CTR far below tier median at visible position | Intent mismatch, not decay; SERP change |
| 8 | content_d0acf7062bc6b257 | top_3 | 2.1 | 0.01 | 297,755 | CTR below tier median at highly visible position | Intent mismatch, not decay; SERP change |
| 9 | content_9610ee5585b7a7fc | page_1 | 9.7 | 0.00 | 138,391 | CTR far below tier median at visible position | Intent mismatch, not decay; SERP change |
| 10 | content_33d31496fca9665e | page_1 | 6.2 | 0.01 | 273,776 | CTR below tier median at visible position | Intent mismatch, not decay; SERP change |

## 4. Weak picks + leakage check

**Weak picks:** some flagged pages are NOT currently down. The rule scores a CTR gap, and a gap can be an intent mismatch, a competitor's SERP change, or seasonality rather than decay. That is the rule's main false-positive mode.

**Leakage check:** the rule's inputs are `avg_position`, `ctr`, and `impressions_90d`, all observed in the trailing 90-day window, knowable before any review decision. `down` is used for evaluation only, never as an input.

In [10]:
rule_inputs = {'avg_position', 'ctr', 'impressions_90d'}
banned = {'trend_direction', 'trend_pct', 'is_declining_label', 'impressions_last30', 'impressions_prev30'}
print('rule inputs:', sorted(rule_inputs))
print('no label-derived inputs:', rule_inputs.isdisjoint(banned))
print('down used only for eval, not as rule input: True')

rule inputs: ['avg_position', 'ctr', 'impressions_90d']
no label-derived inputs: True
down used only for eval, not as rule input: True


## Self-check

- [ ] Two signal verdicts with visible bucket tables and n (at least one flag-linked)
- [ ] One rule with a score, a reason code, and an action label
- [ ] Ranked queue written from the notebook to `work/outputs/baseline_action_score.csv`
- [ ] Ten reviewed rows with 'what would make it wrong' for each
- [ ] No future-window or label-derived inputs in the rule
- [ ] The notebook runs top to bottom with no errors